# 01 · Data collection and cleaning

**Goal:** a clean, continuous panel of daily closing prices for 44 liquid MOEX shares,
2023-01-01 → 2026-06-01.

* **Source:** [MOEX ISS API](https://www.moex.com/a2193) through [`apimoex`](https://github.com/WLM1ke/apimoex), board `TQBR`.
* **Price field:** `LEGALCLOSEPRICE`, the official closing price. It is steadier than the
  last-trade `CLOSE` and is defined the same way for every share.
* **Output:** `data/prices_clean.csv`

The raw download is cached in `data/prices_raw.csv`, so the notebook runs offline.
Set `REFRESH = True` to download it again.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import Image, display

from moex_network import config, correlation, data, metrics, mst, plotting

plotting.use_style()
pd.set_option("display.precision", 3)

REFRESH = False

In [2]:
if REFRESH or not config.RAW_PRICES.exists():
    raw, failed = data.download_prices()
    raw.to_csv(config.RAW_PRICES)
    print("failed tickers:", failed or "none")
else:
    raw = data.load_prices()

print(f"{raw.shape[0]} trading days x {raw.shape[1]} tickers, "
      f"{raw.index.min().date()} → {raw.index.max().date()}")
print("missing values:", int(raw.isna().sum().sum()))
raw.iloc[:5, :8]

867 trading days x 44 tickers, 2023-01-03 → 2026-06-01
missing values: 0


,SBER,VTBR,CBOM,RENI,GAZP,LKOH,ROSN,NVTK
TRADEDATE,,,,,,,,
2023-01-03,141.65,0.016,6.803,50.00,163.79,4136.0,368.50,1077.2
2023-01-04,141.30,0.016,6.795,49.90,162.76,4123.0,365.00,1060.4
2023-01-05,141.31,0.016,6.792,50.06,161.25,4085.0,367.70,1054.8
2023-01-06,141.14,0.016,6.786,50.14,162.00,4098.0,367.65,1050.0
2023-01-09,142.43,0.016,6.804,51.14,163.44,4100.5,369.10,1050.0


## Data-quality checks

### Price jumps

A daily move of more than 20 % in log terms is rare enough that each case needs a look.
It can be a stock split, a large dividend gap, or a data error.

In [3]:
jumps = data.find_price_jumps(raw, threshold=0.2)
jumps

,date,ticker,log_return,price_ratio
0,2023-01-24,NKNC,0.234,1.263
1,2023-04-07,FESH,0.201,1.223
2,2023-08-29,WTCM,0.215,1.239
3,2024-04-08,GMKN,-4.589,0.010
4,2024-07-15,VTBR,8.445,4651.104
5,2024-08-22,BELU,-1.874,0.153
6,2024-08-30,FESH,-0.269,0.764
7,2025-03-27,PLZL,-2.308,0.099
8,2025-07-11,VTBR,-0.255,0.775
9,2026-04-13,CBOM,0.456,1.578


Four of these jumps are **stock splits**. The price ratio matches the split ratio, and
`LEGALCLOSEPRICE` is *not* adjusted for corporate actions:

| Ticker | Date | Event |
|---|---|---|
| GMKN | 2024-04-08 | 1 : 100 split |
| VTBR | 2024-07-15 | 5000 : 1 reverse split |
| BELU | 2024-08-22 | 1 : 8 split |
| PLZL | 2025-03-27 | 1 : 10 split |

Left in the data, each split adds a single return of size |ln 100| ≈ 4.6 or more, up to
ln 5000 ≈ 8.5 for VTBR. Pearson correlation is very sensitive to such outliers, so these
four stocks end up looking almost uncorrelated with the market.

The other jumps (VTBR 2025-07-11, CBOM 2026-04-13 and the smaller ones) are genuine
market moves, such as dividend gaps and news, and are kept.

### Trading halts

Each split was preceded by a few days of suspended trading. During a halt the price
stays the same, which shows up as runs of identical closes:

In [4]:
halts = data.flat_runs(raw, min_run=3)
halts

,ticker,start,end,days
0,VTBR,2024-07-09,2024-07-12,4
1,RENI,2026-03-31,2026-04-02,3
2,GMKN,2024-04-01,2024-04-05,5
3,PLZL,2025-03-24,2025-03-26,3
4,BELU,2024-08-16,2024-08-21,4
5,ABRD,2023-10-31,2023-11-02,3
6,ABRD,2026-02-19,2026-02-24,3
7,ABRD,2026-05-11,2026-05-13,3
8,MTSS,2026-04-06,2026-04-08,3
9,WTCM,2024-07-10,2024-07-12,3


Short runs of 3–6 days are ordinary halts, including the ones right before the four
splits. Three stocks had long suspensions: **ETLN** (50 days, 2025), **OZON** (35 days,
2025) and **MDMG** (17 days, 2024). During those periods their returns are zero, which
pulls their correlations slightly towards zero. This is listed as a limitation in the
README and does not change the structure of the tree.

## Split adjustment

Prices before each split date are multiplied by the split factor (`config.SPLITS`), so
every series is in post-split units.

In [5]:
pd.DataFrame(config.SPLITS)

,ticker,date,factor,note
0,GMKN,2024-04-08,0.010,1:100 split
1,VTBR,2024-07-15,5000.000,5000:1 reverse split
2,BELU,2024-08-22,0.125,1:8 split
3,PLZL,2025-03-27,0.100,1:10 split


In [6]:
adjusted = data.adjust_for_splits(raw)
data.find_price_jumps(adjusted, threshold=0.2)

,date,ticker,log_return,price_ratio
0,2023-01-24,NKNC,0.234,1.263
1,2023-04-07,FESH,0.201,1.223
2,2023-08-29,WTCM,0.215,1.239
3,2024-08-22,BELU,0.205,1.228
4,2024-08-30,FESH,-0.269,0.764
5,2025-07-11,VTBR,-0.255,0.775
6,2026-04-13,CBOM,0.456,1.578


Only the genuine moves are left. The table below shows what the adjustment changes for
the four affected stocks: their average correlation with the other 43 shares.

In [7]:
def mean_corr(prices):
    c = correlation.correlation_matrix(correlation.log_returns(prices))
    return (c.sum() - 1) / (len(c) - 1)

affected = [s["ticker"] for s in config.SPLITS]
pd.DataFrame({
    "mean ρ, unadjusted": mean_corr(raw)[affected],
    "mean ρ, split-adjusted": mean_corr(adjusted)[affected],
})

,"mean ρ, unadjusted","mean ρ, split-adjusted"
GMKN,0.032,0.440
VTBR,-0.014,0.406
BELU,0.116,0.288
PLZL,0.099,0.291


## Cleaning

1. Drop tickers with less than 50 % of days observed (recent IPOs).
2. Drop days on which less than 80 % of tickers traded.
3. Start the sample on the first date when every ticker has a price, so no history is invented.
4. Forward-fill any remaining isolated gaps.

The 44 tickers were picked for having a full history over the period, so these filters
are a safeguard here and change nothing. They matter if the universe is changed in
`config.py`.

In [8]:
clean, report = data.clean_prices(adjusted)
clean.to_csv(config.CLEAN_PRICES)
report

{'shape_before': (867, 44),
 'dropped_tickers': [],
 'common_start': Timestamp('2023-01-03 00:00:00'),
 'shape_after': (867, 44)}